# Energy Consumption Regression Analysis

This notebook analyzes the energy consumption of various energy types based on the average output tokens per prompt using polynomial regression models. The steps involve loading the data, transforming it, fitting the regression models, predicting values, and visualizing the results.


## 1. Load Data

First, we load the data from a CSV file into a pandas DataFrame.

In [12]:
import pandas as pd
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import numpy as np
import matplotlib.pyplot as plt

import altair as alt

from IPython.display import display, Math

In [13]:
# Function to load CSV data
def load_csv_data(file_path):
    """
    Load data from a CSV file into a pandas DataFrame.
    
    Parameters:
        file_path (str): The path to the CSV file.
    
    Returns:
        pd.DataFrame: The loaded DataFrame.
    """
    df = pd.read_csv(file_path)
    return df

In [14]:
# Specify the path to your CSV file
file_path = 'data/emission_regression.csv'

# Load the data into a DataFrame
df_vllm_emission_regression = load_csv_data(file_path)
df_vllm_emission_regression = df_vllm_emission_regression[df_vllm_emission_regression['test_type'] == 'Input-tok']

# Display the first few rows of the DataFrame
df_vllm_emission_regression

,test_type,model_type,parameters,num_examples,num_prompts,total_out_tok,total_in_tok,avg_out_tok,avg_in_tok,actual_emissions_per_10k_prompts,actual_cpu_energy_per_10k_prompts,actual_gpu_energy_per_10k_prompts,actual_ram_energy_per_10k_prompts,pred_emissions_per_10k_prompts,pred_cpu_energy_per_10k_prompts,pred_gpu_energy_per_10k_prompts,pred_ram_energy_per_10k_prompts,actual_emissions_per_1M_out_tok
5,Input-tok,llama3,8.0,1.0,150,2701.0,23799.0,18.007,158.66,1.333196,0.349343,1.095836,0.559783,1.302275,1.302275,1.302275,1.302275,NaN
6,Input-tok,llama3,8.0,5.0,150,2706.0,33399.0,18.040,222.66,1.355123,0.349796,1.127602,0.560540,1.324134,1.324134,1.324134,1.324134,NaN
7,Input-tok,llama3,8.0,10.0,150,2691.0,45249.0,17.940,301.66,1.367448,0.349663,1.146506,0.560304,1.351116,1.351116,1.351116,1.351116,NaN
8,Input-tok,llama3,8.0,20.0,150,2707.0,69399.0,18.047,462.66,1.410753,0.360203,1.184168,0.577226,1.406106,1.406106,1.406106,1.406106,NaN
9,Input-tok,llama3,8.0,30.0,150,2705.0,94149.0,18.033,627.66,1.459989,0.368341,1.237068,0.590234,1.462462,1.462462,1.462462,1.462462,NaN
10,Input-tok,llama3,8.0,70.0,150,2736.0,201549.0,18.240,1343.66,1.654524,0.411301,1.417782,0.659116,1.707012,1.707012,1.707012,1.707012,NaN
11,Input-tok,llama3,8.0,210.0,150,2777.0,561549.0,18.513,3743.66,2.433074,0.589927,2.123813,0.945303,2.526734,2.526734,2.526734,2.526734,NaN
12,Input-tok,llama3,8.0,350.0,150,2821.0,921549.0,18.807,6143.66,3.412185,0.812555,3.016946,1.302004,3.346455,3.346455,3.346455,3.346455,NaN


## 2. Data Transformation

Convert energy values from kilowatt-hours (kWh) to watt-hours (Wh) for better granularity, and calculate prompts per second.

In [15]:
# Transform energy values from kWh to Wh

df_vllm_emission_regression['ram_energy_10k_prompts_Wh'] = df_vllm_emission_regression['actual_ram_energy_per_10k_prompts'] * 1000
df_vllm_emission_regression['gpu_energy_10k_prompts_Wh'] = df_vllm_emission_regression['actual_gpu_energy_per_10k_prompts'] * 1000
df_vllm_emission_regression['cpu_energy_10k_prompts_Wh'] = df_vllm_emission_regression['actual_cpu_energy_per_10k_prompts'] * 1000
df_vllm_emission_regression['total_energy_10k_prompts_Wh'] = df_vllm_emission_regression['cpu_energy_10k_prompts_Wh'] + df_vllm_emission_regression['gpu_energy_10k_prompts_Wh'] + df_vllm_emission_regression['ram_energy_10k_prompts_Wh']

df_vllm_emission_regression = df_vllm_emission_regression[['test_type', 
                                                           'model_type', 
                                                           'parameters',
                                                           'num_examples', 
                                                           'num_prompts', 
                                                           'total_out_tok', 
                                                           'total_in_tok', 
                                                           'avg_out_tok', 
                                                           'avg_in_tok', 
                                                           'total_energy_10k_prompts_Wh', 
                                                           'ram_energy_10k_prompts_Wh', 
                                                           'gpu_energy_10k_prompts_Wh', 
                                                           'cpu_energy_10k_prompts_Wh']]

# Display the updated DataFrame
df_vllm_emission_regression

,test_type,model_type,parameters,num_examples,num_prompts,total_out_tok,total_in_tok,avg_out_tok,avg_in_tok,total_energy_10k_prompts_Wh,ram_energy_10k_prompts_Wh,gpu_energy_10k_prompts_Wh,cpu_energy_10k_prompts_Wh
5,Input-tok,llama3,8.0,1.0,150,2701.0,23799.0,18.007,158.66,2004.962652,559.783065,1095.836393,349.343194
6,Input-tok,llama3,8.0,5.0,150,2706.0,33399.0,18.040,222.66,2037.937525,560.539925,1127.601672,349.795929
7,Input-tok,llama3,8.0,10.0,150,2691.0,45249.0,17.940,301.66,2056.472943,560.304109,1146.505587,349.663248
8,Input-tok,llama3,8.0,20.0,150,2707.0,69399.0,18.047,462.66,2121.597653,577.226306,1184.167966,360.203382
9,Input-tok,llama3,8.0,30.0,150,2705.0,94149.0,18.033,627.66,2195.643321,590.234287,1237.068022,368.341013
10,Input-tok,llama3,8.0,70.0,150,2736.0,201549.0,18.240,1343.66,2488.199355,659.116400,1417.781841,411.301114
11,Input-tok,llama3,8.0,210.0,150,2777.0,561549.0,18.513,3743.66,3659.043010,945.303169,2123.812718,589.927123
12,Input-tok,llama3,8.0,350.0,150,2821.0,921549.0,18.807,6143.66,5131.504626,1302.003665,3016.946062,812.554899


## 3. Polynomial Regression Model Fitting
Fit polynomial regression models for each type of energy consumption using the average output tokens per prompt as the feature variable.

In [16]:
def fit_polynomial_regression(X, y, degree=2):
    polynomial_features = PolynomialFeatures(degree=degree)
    linear_regression = LinearRegression()
    model = make_pipeline(polynomial_features, linear_regression)
    model.fit(X, y)
    return model

In [17]:
# Define the feature variable
X = df_vllm_emission_regression[['avg_in_tok']]

In [18]:
# Fit models for each energy consumption type
models = {}
energy_types = [
    'total_energy_10k_prompts_Wh', 
    'ram_energy_10k_prompts_Wh', 
    'gpu_energy_10k_prompts_Wh', 
    'cpu_energy_10k_prompts_Wh',
]

In [19]:
for energy_type in energy_types:
    y = df_vllm_emission_regression[energy_type]
    models[energy_type] = fit_polynomial_regression(X, y)
    coefs = models[energy_type].named_steps['linearregression'].coef_
    intercept = models[energy_type].named_steps['linearregression'].intercept_
    

## 4. Display Model Coefficients
Display the coefficients of the polynomial regression models for each type of energy consumption.

In [20]:
def display_model_coefficients(model, energy_type):
    coefs = model.named_steps['linearregression'].coef_
    intercept = model.named_steps['linearregression'].intercept_

    # Format the coefficients to 4 decimal places for readability
    coefs = np.round(coefs, 5)
    intercept = np.round(intercept, 5)
    
    
    print("="*20 + f" Regression for {energy_type} " + "="*20 + "\n")
    
    # Print raw coefficients to check their values
    print(f"Raw coefficients:\n intercept={intercept}, coefs={coefs}\n")
    
    print("Formula:")
    # Generate the LaTeX formula
    latex_formula = (
        f"\\hat{{y}} = {intercept:.5f} + {coefs[1]:.5f} x + {coefs[2]:.5f} x^2"
    )

    # Display the LaTeX formula
    display(Math(latex_formula))

    print("\n\n")

In [21]:
# Display coefficients for each energy consumption type
for energy_type in energy_types:
    display_model_coefficients(models[energy_type], energy_type)

==================== Regression for total_energy_10k_prompts_Wh ====================

Raw coefficients:
 intercept=1950.85816, coefs=[0.0000e+00 3.6329e-01 3.0000e-05]

Formula:


<IPython.core.display.Math object>




==================== Regression for ram_energy_10k_prompts_Wh ====================

Raw coefficients:
 intercept=539.62914, coefs=[0.000e+00 8.136e-02 1.000e-05]

Formula:


<IPython.core.display.Math object>




==================== Regression for gpu_energy_10k_prompts_Wh ====================

Raw coefficients:
 intercept=1074.47257, coefs=[0.0000e+00 2.3116e-01 1.0000e-05]

Formula:


<IPython.core.display.Math object>




==================== Regression for cpu_energy_10k_prompts_Wh ====================

Raw coefficients:
 intercept=336.75644, coefs=[0.      0.05077 0.     ]

Formula:


<IPython.core.display.Math object>

## 5. Predict Values
Define x values for prediction and predict the corresponding energy consumption values using the fitted models.

In [37]:
# Define the x values for prediction
predicted_values = {'avg_in_tok': [10, 50, 250, 500, 1000, 1500, 2000, 3000, 4000, 5000, 10000]}

x_values = pd.DataFrame(predicted_values)

# Predict values ensuring feature names are consistent
for energy_type in energy_types:
    model = models[energy_type]
    predicted_values[energy_type] = model.predict(x_values)

# Display the predicted values
predicted_values_df = pd.DataFrame(predicted_values)
predicted_values_df

,avg_in_tok,total_energy_10k_prompts_Wh,ram_energy_10k_prompts_Wh,gpu_energy_10k_prompts_Wh,cpu_energy_10k_prompts_Wh
0,10,1954.493568,540.443482,1076.785550,337.264537
1,50,1969.085426,543.714798,1086.064992,339.305635
2,250,2043.249753,560.406522,1133.122833,349.720398
3,500,2138.779463,582.056661,1193.493476,363.229325
4,1000,2339.253225,627.975222,1319.395907,391.882096
5,1500,2552.279440,677.384824,1452.179861,422.714755
6,2000,2777.858110,730.285469,1591.845340,455.727301
7,3000,3266.672813,846.559886,1891.820870,528.292057
8,4000,3805.697333,976.798473,2219.322497,609.576363
9,5000,4394.931670,1121.001229,2574.350220,699.580221


## 6. Combine Actual and Predicted Data
Combine the actual and predicted data into a single DataFrame for visualization.

In [38]:
# Combine actual and predicted data into a single DataFrame
data = []
for energy_type in energy_types:
    for index, row in df_vllm_emission_regression.iterrows():
        data.append({'avg_in_tok': row['avg_in_tok'], 'Energy_Consumption': row[energy_type], 'Type': 'Actual', 'Energy_Type': energy_type})
    for i, x in enumerate(x_values['avg_in_tok']):
        data.append({'avg_in_tok': x, 'Energy_Consumption': predicted_values[energy_type][i], 'Type': 'Predicted', 'Energy_Type': energy_type})

combined_df = pd.DataFrame(data)
combined_df

,avg_in_tok,Energy_Consumption,Type,Energy_Type
0,158.66,2004.962652,Actual,total_energy_10k_prompts_Wh
1,222.66,2037.937525,Actual,total_energy_10k_prompts_Wh
2,301.66,2056.472943,Actual,total_energy_10k_prompts_Wh
3,462.66,2121.597653,Actual,total_energy_10k_prompts_Wh
4,627.66,2195.643321,Actual,total_energy_10k_prompts_Wh
...,...,...,...,...
71,2000.00,455.727301,Predicted,cpu_energy_10k_prompts_Wh
72,3000.00,528.292057,Predicted,cpu_energy_10k_prompts_Wh
73,4000.00,609.576363,Predicted,cpu_energy_10k_prompts_Wh
74,5000.00,699.580221,Predicted,cpu_energy_10k_prompts_Wh


## 7. Visualize Results
Create an Altair chart to visualize the actual and predicted energy consumption values.

In [39]:
# Create Altair chart
base = alt.Chart(combined_df[combined_df['Type'] == 'Actual']).mark_point(size=100, filled=True).encode(
    x=alt.X('avg_in_tok', title='Average Input Tokens per Prompt'),
    y=alt.Y('Energy_Consumption', title='Energy Consumption (Wh)'),
    color=alt.Color('Energy_Type', title='Energy Type'),
    tooltip=['avg_in_tok', 'Energy_Consumption', 'Energy_Type', 'Type']
).properties(
    width=1200,
    height=600
)


# Highlight predicted values
predicted = alt.Chart(combined_df[combined_df['Type'] == 'Predicted']).mark_point(size=10, filled=False).encode(
    x=alt.X('avg_in_tok', title='Average Input Tokens per Prompt'), 
    y=alt.Y('Energy_Consumption', title='Energy Consumption (Wh)'),
    color=alt.Color('Energy_Type', title='Energy Type'),
    tooltip=['avg_in_tok', 'Energy_Consumption', 'Energy_Type']
)

regression = predicted.transform_regression('avg_in_tok', 'Energy_Consumption', groupby=['Energy_Type'], method="quad").mark_line()

# Combine charts
final_chart = base + regression + predicted

# Display the chart in Streamlit
final_chart

alt.LayerChart(...)

In [48]:
import pandas as pd
import numpy as np

def calculate_energy_change(model, start_tokens, end_tokens):
    """
    Calculate the percentage change in energy consumption when changing the number of input tokens.

    Parameters:
    - model: The scikit-learn model used for prediction.
    - start_tokens (int): The initial number of input tokens.
    - end_tokens (int): The new number of input tokens.

    Returns:
    - results (dict): A dictionary containing the token counts, predicted energies, and percentage change range.
    """
    # Create a DataFrame with the start and end token counts
    x_values = pd.DataFrame({'avg_in_tok': [start_tokens, end_tokens]})
    
    # Predict the energy consumption for both token counts
    y_values = model.predict(x_values)
    
    # Extract the predicted energy values
    energy_start = y_values[0]
    energy_end = y_values[1]
    
    # Calculate the percentage change in energy consumption
    energy_change_percent = ((energy_end - energy_start) / energy_start) * 100
    
    # Calculate the factor change in energy consumption
    energy_change_factor = energy_end / energy_start if energy_start != 0 else np.inf
    
    # Calculate the next lower and higher multiples of 50% for percentage change
    lower_percent = np.floor(energy_change_percent / 50) * 50
    upper_percent = np.ceil(energy_change_percent / 50) * 50

    # Calculate the next lower and higher multiples of 0.5x for factor change
    lower_factor = np.floor(energy_change_factor / 0.5) * 0.5
    upper_factor = np.ceil(energy_change_factor / 0.5) * 0.5

    # Handle percentage change range formatting
    if energy_change_percent == 0:
        energy_change_range = "No Change"
    elif energy_change_percent < 0:
        energy_change_range = f"Decrease of {abs(energy_change_percent):.2f}%"
    else:
        # Cap the upper bound at a maximum value if desired
        max_upper_bound = 10000  # You can adjust this value as needed
        if upper_percent > max_upper_bound:
            energy_change_range = f">{int(max_upper_bound)}%"
        else:
            if lower_percent == upper_percent:
                energy_change_range = f"{int(upper_percent)}%"
            else:
                energy_change_range = f"{int(lower_percent)}% – {int(upper_percent)}%"

    # Handle factor change range formatting
    if energy_change_factor == 1:
        energy_change_factor_range = "No Change"
    elif energy_change_factor < 1:
        energy_change_factor_range = f"{energy_change_factor:.2f}x"
    else:
        # Cap the upper bound at a maximum value if desired
        max_upper_factor = 1000  # You can adjust this value as needed
        if upper_factor > max_upper_factor:
            energy_change_factor_range = f">{max_upper_factor}x"
        else:
            if lower_factor == upper_factor:
                energy_change_factor_range = f"{upper_factor:.1f}x"
            else:
                energy_change_factor_range = f"{lower_factor:.1f}x – {upper_factor:.1f}x"

    # Prepare the results dictionary
    results = {
        'start_tokens': start_tokens,
        'end_tokens': end_tokens,
        'energy_start': energy_start,
        'energy_end': energy_end,
        'energy_change_percent': energy_change_percent,
        'energy_change_range': energy_change_range,
        'energy_change_factor': energy_change_factor,
        'energy_change_factor_range': energy_change_factor_range
    }
    
    return results


In [49]:
# Define the start and end token counts
start_tokens = 1000
end_tokens = 10000

# Call the function to calculate the energy change
results = calculate_energy_change(models['total_energy_10k_prompts_Wh'], start_tokens, end_tokens)

# Print the results in a nicely formatted way
print(f"Energy Consumption Analysis:\n")
print(f"- Start Tokens: {results['start_tokens']}")
print(f"- End Tokens: {results['end_tokens']}\n")
print(f"- Energy Consumption at Start: {results['energy_start']:.2f} Wh")
print(f"- Energy Consumption at End: {results['energy_end']:.2f} Wh\n")
print(f"Percentage Change in Energy Consumption: {results['energy_change_range']}")
print(f"Factor Change in Energy Consumption: {results['energy_change_factor_range']}")


Energy Consumption Analysis:

- Start Tokens: 1000
- End Tokens: 10000

- Energy Consumption at Start: 2339.25 Wh
- Energy Consumption at End: 8094.25 Wh

Percentage Change in Energy Consumption: 200% – 250%
Factor Change in Energy Consumption: 3.0x – 3.5x
